# w9_l40.ipynb — SMALL jobs (all g512 fixed-split arms + all 5-fold CV)

Job table + labels live in `Pod/w9_jobs.py` (shared with `w9_a100.ipynb`).
Labels are IDENTICAL to the retired `w9_all.ipynb`: claims + result files
interoperate. This notebook never builds the full pool — it waits for the
READY marker (builder = w9_a100.ipynb). Run top to bottom; re-run after any
interruption — done jobs skip.


In [ ]:
# paths (tree-specific) + flags. Job table / budgets live in Pod/w9_jobs.py.
REPO = os.path.abspath("..")   # this release folder (contains Pod/ and VICReg_review/)
DATA_SRC = "/workspace/fusion_cache_w9"
DATA_RAM = "/dev/shm/fusion_cache_w9"
OUT_FS = "/workspace/w9_out"        # fixed-split results
OUT_CV = "/workspace/w9_cv_out"     # CV results


In [ ]:
# Local setup (release build: the code ships with this folder -- no
# repository synchronisation is needed or performed).
import importlib.util
import os
import sys
for pkg in ("sklearn", "scipy"):
    if importlib.util.find_spec(pkg) is None:
        %pip -q install scikit-learn scipy
        break
os.chdir(REPO)
sys.path.insert(0, REPO)
sys.path.insert(0, os.path.join(REPO, "Pod"))
import w9_jobs as J
print("machinery loaded")

In [ ]:
# shared job table + queue machinery (labels compatible with old w9_all runs)
import sys, importlib
if REPO not in sys.path:
    sys.path.insert(0, REPO)
import Pod.w9_jobs
importlib.reload(Pod.w9_jobs)
from Pod import w9_jobs as J
FULL_POOL = J.FULL_POOL
J.summary()


In [ ]:
# Stage the corpus into RAM.
import shutil
from pathlib import Path
REQUIRED = ["games.npz", "wiki_eval.npz", "wscan_gal_rev.npz",
            "wscan_pool_rev.npy", "wscan_pool_rev_rid.npy", "wscan_pool_rev_len.npy",
            "ss_queries_rev.npz", "ss_queries_rev_S.npy",
            "wiki_clean_views.npz", "wiki_llm_views.npz", "sp_raw_views.npz",
            "tag_labels.npz",
            "wiki_eval_split.json", "_tag_splitM.json"]
src = Path(DATA_SRC)
missing = [f for f in REQUIRED if not (src / f).exists()]
assert not missing, f"missing in {DATA_SRC}: {missing}"
dst = Path(DATA_RAM)
dst.mkdir(parents=True, exist_ok=True)
for f in REQUIRED:
    s, d = src / f, dst / f
    if not d.exists() or d.stat().st_size != s.stat().st_size:
        print(f"staging {f} ({s.stat().st_size/1e9:.2f} GB) ...", flush=True)
        shutil.copyfile(s, d)
# optional prebuilt anchor packs (wscan_gal_rev_g*.npz): stage if present
for s in sorted(src.glob("wscan_gal_rev_g*.npz")):
    d = dst / s.name
    if not d.exists() or d.stat().st_size != s.stat().st_size:
        print(f"staging {s.name} ({s.stat().st_size/1e9:.2f} GB) ...", flush=True)
        shutil.copyfile(s, d)
DATA_DIR = str(dst)
print("corpus in RAM:", DATA_DIR)

In [ ]:
# l40 pods do NOT build the 150 GB full pool -- they wait for the READY
# marker (the builder lives in w9_a100.ipynb; multi-machine safe).
import time
from pathlib import Path
if FULL_POOL:
    ready = Path(DATA_SRC) / "full_pool_READY"
    t0 = time.time()
    while not ready.exists():
        print(f"waiting for full_pool_READY (build it from w9_a100.ipynb) "
              f"[{(time.time()-t0)/60:.0f} min]", flush=True)
        time.sleep(60)
    print("full pool READY:", (Path(DATA_SRC) / "full_pool_fp16.npy"))


In [ ]:
# Stage the 150 GB full pool onto FAST LOCAL storage (thread-parallel copy,
# pattern from Pod/h5_staging.py). Network-volume random reads are slow; one
# sequential parallel copy (~5-15 min) buys RAM/NVMe-speed sampling for the
# whole campaign. Falls back to the volume mmap if no local space is found.
import os, sys
from pathlib import Path
FULL_POOL_PATH = ""
if FULL_POOL:
    if REPO not in sys.path:
        sys.path.insert(0, REPO)
    from Pod.h5_staging import parallel_copy

    src_v = Path(DATA_SRC) / "full_pool_fp16.npy"
    src_m = Path(DATA_SRC) / "full_pool_meta.npz"
    need = src_v.stat().st_size + (5 << 30)

    def _free(p):
        st = os.statvfs(p)
        return st.f_bavail * st.f_frsize

    dest_dir = None
    for cand in ("/dev/shm", "/root/data", "/root"):
        Path(cand).mkdir(parents=True, exist_ok=True)
        if _free(cand) > need:
            dest_dir = Path(cand)
            break
    if dest_dir is None:
        print("WARNING: no local space for the full pool -- workers will mmap "
              "the NETWORK VOLUME copy (slow first pass).")
        FULL_POOL_PATH = str(src_v)
    else:
        dst_v = dest_dir / "full_pool_fp16.npy"
        if dst_v.exists() and dst_v.stat().st_size == src_v.stat().st_size:
            print("local full pool already staged:", dst_v)
        else:
            import time
            t0 = time.time()
            tmp = dst_v.with_name(dst_v.name + ".copying")
            print(f"staging {src_v.stat().st_size/2**30:.0f} GiB -> {dst_v} "
                  f"(8 threads) ...", flush=True)
            parallel_copy(src_v, tmp, workers=8)
            os.replace(tmp, dst_v)
            print(f"staged in {(time.time()-t0)/60:.1f} min", flush=True)
        import shutil
        shutil.copyfile(src_m, dest_dir / "full_pool_meta.npz")
        FULL_POOL_PATH = str(dst_v)
print("FULL_POOL_PATH =", FULL_POOL_PATH or "(disabled)")

In [ ]:
# drain the SMALL job class. SWEEP_BIG=True additionally sweeps the other
# class after this one empties (single-pod full-campaign mode).
SWEEP_BIG = False
fails = J.run_queue("small", repo=REPO, data_dir=DATA_DIR,
                    out_fs=OUT_FS, out_cv=OUT_CV,
                    full_pool_path=FULL_POOL_PATH,
                    sweep_other=SWEEP_BIG)


In [ ]:
# ---- OPTIONAL 续训 (extension): push finished arms past their budget ----
# Runs after the queue drains. The worker rebuilds state from the newest
# checkpoint (weights + mq shadow; fresh opt/rng; the mq queue re-prefills
# from the shadow tower). Old per-epoch probe jsons are reused; the best
# pick is redone over ALL checkpoints. Empty EXTEND = no-op.
EXTEND = [
    # (arm, cap, nsp, lead, wiki_src, view_w, target_epochs)
    # (mq cap-curve extensions live in the dedicated w9_mq_i2ce.ipynb)
    ("wcle_i2cce_icetf", 2048, False, 0, "clean", 16, 2000),
]
for *job6, ep2 in EXTEND:
    J.extend_fs(tuple(job6), ep2, repo=REPO, data_dir=DATA_DIR,
                out_fs=OUT_FS, full_pool_path=FULL_POOL_PATH)


In [ ]:
J.aggregate(OUT_FS, OUT_CV)


In [ ]:
# AUTO-STOP removed in the release build: stopping the machine is cloud-
# provider tooling, not part of the experiment. All results are already on
# the shared volume when the run cells finish.
print("run complete -- results are in", OUT_DIR)